# LOO summary table (DEG 200)

Builds a NeurIPS-ready summary table from `results/loo_summary_deg_200.csv`.

- Aggregates across seeds and held-out cell types (mean ± std).
- Bolds the best performer per metric (direction-aware).
- Renames `baseline` to `mean shift`.
- Annotates columns with ↑ (higher is better) or ↓ (lower is better).

In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

FILE_NAME = "loo_summary_merfish"
CSV_PATH = f"../results/{FILE_NAME}.csv"
OUT_DIR = Path("../results")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, engine="python")
df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,mixing_index,edistance_global,edistance_local,rmse
0,C57BL6J-2.036,baseline,GABAergic neuron_Fiber_tracts,200,0.068864,0.230110,0.115,0.217391,0.025,0.019022,14.386200,14.838829,521.428223
1,C57BL6J-2.036,baseline,GABAergic neuron_Isocortex,200,0.460826,0.497896,0.165,0.696970,0.115,0.293658,23.667892,27.172966,1234.587036
2,C57BL6J-2.036,baseline,astrocyte_Fiber_tracts,200,0.399375,0.309090,0.210,0.857143,0.180,0.069700,17.304278,18.637169,1589.633545
3,C57BL6J-2.036,baseline,astrocyte_Isocortex,200,0.701164,0.642562,0.280,0.928571,0.260,0.000000,36.404189,38.584384,2399.222656
4,C57BL6J-2.036,baseline,endothelial cell_Fiber_tracts,200,0.265527,0.311898,0.185,0.945946,0.175,0.003236,17.183491,17.923578,1034.296997


In [7]:
# Rename holdout_celltype by replacing the last '_' in with '-'
df["holdout_celltype"] = df["holdout_celltype"].str.replace("Fiber_tracts", "Fiber-tracts", regex=False)

In [8]:
# Split holdout_celltype on '_' and place everything in [0] as holdout_celltype, and everything in [1:] as perturbation
df["perturbation"] = df["holdout_celltype"].apply(lambda x: "".join(x.split("_")[-1]) if len(x.split("_")) > 1 else "")
df["holdout_celltype"] = df["holdout_celltype"].apply(lambda x: "-".join(x.split("_")[:-1]))
df

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,mixing_index,edistance_global,edistance_local,rmse,perturbation
0,C57BL6J-2.036,baseline,GABAergic neuron,200,0.068864,0.230110,0.115,0.217391,0.025,0.019022,14.386200,14.838829,521.428223,Fiber-tracts
1,C57BL6J-2.036,baseline,GABAergic neuron,200,0.460826,0.497896,0.165,0.696970,0.115,0.293658,23.667892,27.172966,1234.587036,Isocortex
2,C57BL6J-2.036,baseline,astrocyte,200,0.399375,0.309090,0.210,0.857143,0.180,0.069700,17.304278,18.637169,1589.633545,Fiber-tracts
3,C57BL6J-2.036,baseline,astrocyte,200,0.701164,0.642562,0.280,0.928571,0.260,0.000000,36.404189,38.584384,2399.222656,Isocortex
4,C57BL6J-2.036,baseline,endothelial cell,200,0.265527,0.311898,0.185,0.945946,0.175,0.003236,17.183491,17.923578,1034.296997,Fiber-tracts
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,C57BL6J-2.041,scgen,endothelial cell,200,0.874021,0.910055,0.440,1.000000,0.440,0.337851,2.805977,1.486592,1271.375854,Isocortex
174,C57BL6J-2.041,scgen,glutamatergic neuron,200,0.364032,0.433754,0.120,0.875000,0.105,0.821467,9.585938,10.575816,5985.251465,Fiber-tracts
175,C57BL6J-2.041,scgen,glutamatergic neuron,200,0.580948,0.734886,0.220,1.000000,0.220,0.679148,12.129467,15.351844,8362.841797,Isocortex
176,C57BL6J-2.041,scgen,oligodendrocyte,200,0.302931,0.704027,0.380,0.986842,0.375,0.304157,1.218217,2.155563,2112.147461,Fiber-tracts


In [9]:
# Metrics we want in the table and whether higher (+1) or lower (-1) is better.
METRICS = {
    "spearman": +1,
    "pearson": +1,
    "precision": +1,
    "direction_match_k": +1,
    "edistance_local": -1,
    #"rmse": -1,
}

PRETTY_METRIC = {
    "spearman": r"Spearman $\uparrow$",
    "pearson": r"Pearson $\uparrow$",
    "precision": r"Precision $\uparrow$",
    "direction_match_k": r"Direction Match (K) $\uparrow$",
    "edistance_local": r"E-dist (local) $\downarrow$",
    #"rmse": r"RMSE $\downarrow$",
}

# Model display order + renaming (baseline -> mean shift).
MODEL_RENAME = {
    "baseline": "mean shift",
    "cpa": "CPA",
    "scgen": "scGen",
    #"mintflow": "MintFlow",
    "cellina-ablated": "cellina (ablated)",
    "cellina-graph": "cellina (graph)",
    "cellina": "cellina",
}
MODEL_ORDER = [
    "mean shift",
    "CPA",
    "scGen",
    #"MintFlow",
    "cellina (ablated)",
    "cellina (graph)",
    "cellina",
]

df["model_name"] = df["model_name"].map(lambda x: MODEL_RENAME.get(x, x))
df["model_name"].unique()

array(['mean shift', 'cellina (ablated)', 'cellina', 'cellina (graph)',
       'CPA', 'scGen'], dtype=object)

In [10]:
# Two-step aggregation:
#   1) within each slide, average over held-out cell types
#   2) across slides, compute mean ± std
agg = (
    df.groupby(["perturbation", "model_name"])[list(METRICS)]
    .agg(["mean", "std"])
)

# Ensure a consistent model order within each perturbation. Build a
# MultiIndex with (perturbation, model) in the desired display order and
# reindex the aggregated table to that index.
perturbations = list(df["perturbation"].dropna().unique())
# keep stable ordering; treat empty string as its own group if present
if any(p == "" for p in perturbations):
    perturbations = sorted(perturbations, key=lambda x: (x != "", x))
else:
    perturbations = sorted(perturbations)

desired_index = pd.MultiIndex.from_product(
    [perturbations, MODEL_ORDER], names=["perturbation", "model_name"]
)
agg = agg.reindex(desired_index)

In [11]:
agg

spearman             pearson            \
                                    mean       std      mean       std   
perturbation model_name                                                  
Fiber-tracts mean shift         0.216837  0.202377  0.217113  0.177359   
             CPA                0.680642  0.094093  0.707441  0.134819   
             scGen              0.567599  0.134338  0.621237  0.134904   
             cellina (ablated)  0.684867  0.098927  0.703295  0.132585   
             cellina (graph)    0.731632  0.093846  0.725382  0.148629   
             cellina            0.758496  0.131027  0.758680  0.160978   
Isocortex    mean shift         0.578568  0.164632  0.506310  0.156934   
             CPA                0.840475  0.084598  0.821171  0.173743   
             scGen              0.756041  0.103771  0.758697  0.157755   
             cellina (ablated)  0.852675  0.076493  0.825859  0.164286   
             cellina (graph)    0.892049  0.036857  0.885319  0.128049   
             cellina            0.873039  0.077882  0.852132  0.172570   

                               precision           direction_match_k  \
                                    mean       std              mean   
perturbation model_name                                                
Fiber-tracts mean shift         0.214000  0.043924          0.145000   
             CPA                0.464333  0.053045          0.439000   
             scGen              0.287667  0.092637          0.276333   
             cellina (ablated)  0.436667  0.078323          0.410667   
             cellina (graph)    0.508214  0.055074          0.490000   
             cellina            0.591667  0.078733          0.572333   
Isocortex    mean shift         0.284333  0.056942          0.243333   
             CPA                0.568667  0.062777          0.565333   
             scGen              0.321000  0.072215          0.314333   
             cellina (ablated)  0.558667  0.086364          0.554667   
             cellina (graph)    0.649286  0.050947          0.647143   
             cellina            0.636667  0.077567          0.634667   

                                         edistance_local            
                                     std            mean       std  
perturbation model_name                                             
Fiber-tracts mean shift         0.060651       20.506667  3.101726  
             CPA                0.062940        7.184391  5.091321  
             scGen              0.101585        5.733346  4.349483  
             cellina (ablated)  0.074135        7.252630  5.350933  
             cellina (graph)    0.063002        7.621933  4.931337  
             cellina            0.098830        6.483417  4.403159  
Isocortex    mean shift         0.077912       34.583413  6.696663  
             CPA                0.063681        8.265638  6.092033  
             scGen              0.073481        7.971304  6.325841  
             cellina (ablated)  0.088388        8.623152  6.816974  
             cellina (graph)    0.054375        8.337464  5.552466  
             cellina            0.081097        7.761777  5.473598

In [12]:
def find_best(agg: pd.DataFrame) -> dict:
    best = {}
    if isinstance(agg.index, pd.MultiIndex):
        # agg index: (perturbation, model_name)
        # collect perturbations preserving order
        raw = [p for p, _ in agg.index.tolist()]
        seen = set()
        perturbations = [p for p in raw if not (p in seen or seen.add(p))]
        for metric, direction in METRICS.items():
            for pert in perturbations:
                try:
                    means = agg.loc[pert][(metric, "mean")].dropna()
                except KeyError:
                    continue
                if means.empty:
                    continue
                best[(pert, metric)] = means.idxmax() if direction > 0 else means.idxmin()
    else:
        for metric, direction in METRICS.items():
            means = agg[(metric, "mean")].dropna()
            if means.empty:
                continue
            best[metric] = means.idxmax() if direction > 0 else means.idxmin()
    return best


def format_cell(mean, std, bold, na_str="--"):
    if pd.isna(mean):
        return na_str
    s = f"{mean:.3f} $\\pm$ {std:.3f}" if not pd.isna(std) else f"{mean:.3f}"
    return f"\\textbf{{{s}}}" if bold else s

In [13]:
best = find_best(agg)
table = pd.DataFrame(index=agg.index, columns=[PRETTY_METRIC[m] for m in METRICS])

for metric in METRICS:
    col = PRETTY_METRIC[metric]
    for model in agg.index:
        pert, model_name = model

        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]

        bold = (best.get((pert, metric)) == model_name)

        table.loc[model, col] = format_cell(mean, std, bold)
table

Spearman $\uparrow$  \
perturbation model_name                                      
Fiber-tracts mean shift                  0.217 $\pm$ 0.202   
             CPA                         0.681 $\pm$ 0.094   
             scGen                       0.568 $\pm$ 0.134   
             cellina (ablated)           0.685 $\pm$ 0.099   
             cellina (graph)             0.732 $\pm$ 0.094   
             cellina            \textbf{0.758 $\pm$ 0.131}   
Isocortex    mean shift                  0.579 $\pm$ 0.165   
             CPA                         0.840 $\pm$ 0.085   
             scGen                       0.756 $\pm$ 0.104   
             cellina (ablated)           0.853 $\pm$ 0.076   
             cellina (graph)    \textbf{0.892 $\pm$ 0.037}   
             cellina                     0.873 $\pm$ 0.078   

                                        Pearson $\uparrow$  \
perturbation model_name                                      
Fiber-tracts mean shift                  0.217 $\pm$ 0.177   
             CPA                         0.707 $\pm$ 0.135   
             scGen                       0.621 $\pm$ 0.135   
             cellina (ablated)           0.703 $\pm$ 0.133   
             cellina (graph)             0.725 $\pm$ 0.149   
             cellina            \textbf{0.759 $\pm$ 0.161}   
Isocortex    mean shift                  0.506 $\pm$ 0.157   
             CPA                         0.821 $\pm$ 0.174   
             scGen                       0.759 $\pm$ 0.158   
             cellina (ablated)           0.826 $\pm$ 0.164   
             cellina (graph)    \textbf{0.885 $\pm$ 0.128}   
             cellina                     0.852 $\pm$ 0.173   

                                      Precision $\uparrow$  \
perturbation model_name                                      
Fiber-tracts mean shift                  0.214 $\pm$ 0.044   
             CPA                         0.464 $\pm$ 0.053   
             scGen                       0.288 $\pm$ 0.093   
             cellina (ablated)           0.437 $\pm$ 0.078   
             cellina (graph)             0.508 $\pm$ 0.055   
             cellina            \textbf{0.592 $\pm$ 0.079}   
Isocortex    mean shift                  0.284 $\pm$ 0.057   
             CPA                         0.569 $\pm$ 0.063   
             scGen                       0.321 $\pm$ 0.072   
             cellina (ablated)           0.559 $\pm$ 0.086   
             cellina (graph)    \textbf{0.649 $\pm$ 0.051}   
             cellina                     0.637 $\pm$ 0.078   

                               Direction Match (K) $\uparrow$  \
perturbation model_name                                         
Fiber-tracts mean shift                     0.145 $\pm$ 0.061   
             CPA                            0.439 $\pm$ 0.063   
             scGen                          0.276 $\pm$ 0.102   
             cellina (ablated)              0.411 $\pm$ 0.074   
             cellina (graph)                0.490 $\pm$ 0.063   
             cellina               \textbf{0.572 $\pm$ 0.099}   
Isocortex    mean shift                     0.243 $\pm$ 0.078   
             CPA                            0.565 $\pm$ 0.064   
             scGen                          0.314 $\pm$ 0.073   
             cellina (ablated)              0.555 $\pm$ 0.088   
             cellina (graph)       \textbf{0.647 $\pm$ 0.054}   
             cellina                        0.635 $\pm$ 0.081   

                               E-dist (local) $\downarrow$  
perturbation model_name                                     
Fiber-tracts mean shift                 20.507 $\pm$ 3.102  
             CPA                         7.184 $\pm$ 5.091  
             scGen              \textbf{5.733 $\pm$ 4.349}  
             cellina (ablated)           7.253 $\pm$ 5.351  
             cellina (graph)             7.622 $\pm$ 4.931  
             cellina                     6.483 $\pm$ 4.403  
Isocortex    mean shift          

In [14]:
def insert_midrule_perts(latex, table):
    groups = table.index.get_level_values(0)
    boundaries = [i for i in range(1, len(groups)) if groups[i] != groups[i - 1]]

    lines = latex.splitlines()
    new_lines = []

    row_idx = 0

    for line in lines:
        new_lines.append(line)

        # detect data rows (skip header)
        if "&" in line and "\\\\" in line:
            if row_idx in boundaries:
                new_lines.append(r"\midrule")
            row_idx += 1

    latex = "\n".join(new_lines)
    return latex

In [15]:
is_multi_pert = table.index.get_level_values(0).nunique() > 1

if is_multi_pert:
    latex = table.to_latex(
        index_names=False, # removes the extra header row
        escape=False,
        header=False,
        column_format="ll" + "c" * table.shape[1],  # two index levels → 'll'
        multirow=True,  # for multiple perturbations
        caption=(
            f"Leave-one-celltype-out performance (top {df.n_deg.unique()[0]} DEGs). For each slide we "
            "first average over the held-out cell types, then report mean $\\pm$ std "
            f"across {df['sid'].nunique()} slides. Best per metric in \\textbf{{bold}}."
        ),
        label=f"tab:{FILE_NAME}",
    )

    latex = latex.replace(r"\cline{1-7}", "")
    latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")
    latex = latex.replace(r"\multirow[t]", r"\multirow[c]")
    header = "Holdout \\\ perturbation & Method & " + " & ".join(table.columns) + r" \\"
    latex = latex.replace(
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}",
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}\n\\toprule\n" + header
    )
    latex = latex.replace(r"\midrule", "")
    latex = insert_midrule_perts(latex, table)
else:
    flat = table.reset_index(level=0, drop=True)

    latex = flat.to_latex(
        index=True,
        index_names=False,
        escape=False,
        column_format="l" + "c" * flat.shape[1],
        caption=(
            f"Leave-one-celltype-out performance (top {df.n_deg.unique()[0]} DEGs). "
            "Mean $\\pm$ std across slides. Best per metric in \\textbf{bold}."
        ),
        label=f"tab:{FILE_NAME}",
    )

print(latex)

(OUT_DIR / f"{FILE_NAME}_table.tex").write_text(latex)

\begin{table}[t]
\centering
\caption{Leave-one-celltype-out performance (top 200 DEGs). For each slide we first average over the held-out cell types, then report mean $\pm$ std across 3 slides. Best per metric in \textbf{bold}.}
\label{tab:loo_summary_merfish}
\begin{tabular}{llccccc}
\toprule
Holdout \\ perturbation & Method & Spearman $\uparrow$ & Pearson $\uparrow$ & Precision $\uparrow$ & Direction Match (K) $\uparrow$ & E-dist (local) $\downarrow$ \\
\toprule

\multirow[c]{6}{*}{Fiber-tracts} & mean shift & 0.217 $\pm$ 0.202 & 0.217 $\pm$ 0.177 & 0.214 $\pm$ 0.044 & 0.145 $\pm$ 0.061 & 20.507 $\pm$ 3.102 \\
 & CPA & 0.681 $\pm$ 0.094 & 0.707 $\pm$ 0.135 & 0.464 $\pm$ 0.053 & 0.439 $\pm$ 0.063 & 7.184 $\pm$ 5.091 \\
 & scGen & 0.568 $\pm$ 0.134 & 0.621 $\pm$ 0.135 & 0.288 $\pm$ 0.093 & 0.276 $\pm$ 0.102 & \textbf{5.733 $\pm$ 4.349} \\
 & cellina (ablated) & 0.685 $\pm$ 0.099 & 0.703 $\pm$ 0.133 & 0.437 $\pm$ 0.078 & 0.411 $\pm$ 0.074 & 7.253 $\pm$ 5.351 \\
 & cellina (graph) & 0.73

2069

In [16]:
# Markdown preview (arrows rendered as unicode so they look right in the notebook).
md_metric = {
    "spearman": "Spearman ↑",
    "pearson": "Pearson ↑",
    "precision": "Precision ↑",
    "direction_match_k": "Direction Match (K) ↑",
    "edistance_local": "E-dist (local) ↓",
    #"rmse": "RMSE ↓",
}

md_table = pd.DataFrame(index=agg.index, columns=[md_metric[m] for m in METRICS])
for metric in METRICS:
    col = md_metric[metric]
    for model in agg.index:
        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]
        if pd.isna(mean):
            md_table.loc[model, col] = "--"
            continue
        s = f"{mean:.3f} ± {std:.3f}"
        md_table.loc[model, col] = f"**{s}**" if best.get(metric) == model else s

md_table.index.name = "Method"
print(md_table.to_markdown())

| Method                                | Spearman ↑    | Pearson ↑     | Precision ↑   | Direction Match (K) ↑   | E-dist (local) ↓   |
|:--------------------------------------|:--------------|:--------------|:--------------|:------------------------|:-------------------|
| ('Fiber-tracts', 'mean shift')        | 0.217 ± 0.202 | 0.217 ± 0.177 | 0.214 ± 0.044 | 0.145 ± 0.061           | 20.507 ± 3.102     |
| ('Fiber-tracts', 'CPA')               | 0.681 ± 0.094 | 0.707 ± 0.135 | 0.464 ± 0.053 | 0.439 ± 0.063           | 7.184 ± 5.091      |
| ('Fiber-tracts', 'scGen')             | 0.568 ± 0.134 | 0.621 ± 0.135 | 0.288 ± 0.093 | 0.276 ± 0.102           | 5.733 ± 4.349      |
| ('Fiber-tracts', 'cellina (ablated)') | 0.685 ± 0.099 | 0.703 ± 0.133 | 0.437 ± 0.078 | 0.411 ± 0.074           | 7.253 ± 5.351      |
| ('Fiber-tracts', 'cellina (graph)')   | 0.732 ± 0.094 | 0.725 ± 0.149 | 0.508 ± 0.055 | 0.490 ± 0.063           | 7.622 ± 4.931      |
| ('Fiber-tracts', 'cellina')           |